In [3]:
!pip	install	-q	transformers	datasets	accelerate	sentencepiece	evaluate	sacrebleu	opencc-python-reimplemented	pandas	torch	hanzidentifier


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
OUTPUT_DIR = './en-zhtw-model'
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU found")

True
NVIDIA GeForce RTX 5060 Laptop GPU


In [2]:
from datasets import load_dataset
from opencc import OpenCC
import pandas as pd

cc = OpenCC('s2t')
MAX_TRAIN = 200_000 # lower this (e.g. 20_000) for a faster first run
VAL_SIZE = 2000
print('Downloading OPUS-100 en-zh ...')
raw_ds = load_dataset('Helsinki-NLP/opus-100', 'en-zh')

def to_rows(split, limit):
    rows = []
    for ex in split.select(range(min(limit, len(split)))):
        pair = ex['translation']
        en, zh = pair['en'].strip(), pair['zh'].strip()
        if en and zh:
            rows.append({'en': en, 'zh_hant': cc.convert(zh)})
    return rows

train_rows = to_rows(raw_ds['train'], MAX_TRAIN)
val_rows = to_rows(raw_ds['validation'], VAL_SIZE)

pd.DataFrame(train_rows).to_csv('train.csv', index=False)
pd.DataFrame(val_rows).to_csv('val.csv', index=False)
print(f'{len(train_rows)} train rows, {len(val_rows)} val rows')


C:\Users\Dheeransh\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\Users\Dheeransh\AppData\Roaming\Python\Python314\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Dheeransh\.cache\huggingface\hub\datasets--Helsinki-NLP--opus-100. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating validation split: 100%|██████████| 2000/2000 [00:00<00:00, 236718.91 examples/s]


200000 train rows, 2000 val rows


In [5]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = 'facebook/nllb-200-distilled-600M'
SRC_LANG = 'eng_Latn'
TGT_LANG = 'zho_Hant'
MAX_LEN = 128

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

OSError: Can't load the model for 'facebook/nllb-200-distilled-600M'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'facebook/nllb-200-distilled-600M' is the correct path to a directory containing a file named pytorch_model.bin.